# Using GPT-5 - OpenAI API Guide

## Introduction to OpenAI's Most Intelligent Model

GPT-5 is OpenAI's most advanced reasoning model, specifically trained for:
- **Code generation**, bug fixing, and refactoring
- **Instruction following** with high accuracy
- **Long context** handling and **tool calling** for agentic tasks

This notebook demonstrates GPT-5's key features with practical code examples.

> **Model update (GPT-5.5 refresh).** All `model=` calls in this notebook now target **`gpt-5.5`**, the current frontier model for coding and professional work. `reasoning_effort` is pinned explicitly in cells that care about cost/latency, because the default moved across the 5.x line (`medium` -> `none` -> `medium`); GPT-5.5 defaults to `medium`.
>
> **`gpt-5.5-instant`** is available in the API. Light, latency-sensitive demo cells can use it — the model string is `gpt-5.5-instant`. This notebook uses `gpt-5.5` throughout for consistency.

## Setup

Install the OpenAI Python client if you haven't already:

In [ ]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("OPENAI_API_KEY")

In [ ]:
# Install OpenAI client
# Run this if you're running this notebook on google colab
!pip install openai

In [1]:
from openai import OpenAI
import os

# Initialize client
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")  # Set your API key as environment variable
)

## GPT-5.5 Model Variants (current line)

The GPT-5.5 line replaces the original GPT-5 family. Prices are per 1M tokens, standard tier (researched 2026-06-05).

| Model | Input | Output | Cached input | Context | Max output | Best for |
|-------|------:|-------:|-------------:|--------:|-----------:|----------|
| **`gpt-5.5`** | $5.00 | $30.00 | $0.50 | 1,050,000 | 128,000 | Frontier coding + professional work (course default) |
| **`gpt-5.5-pro`** | $30.00 | $180.00 | — | 1,050,000 | 128,000 | Hardest problems needing extended reasoning (use background mode) |
| **`gpt-5.5-instant`** | $5.00 | $30.00 | — | ~1M | 128,000 | Fast, concise everyday tasks; ChatGPT default variant |
| **`gpt-5.4-mini`** | $0.75 | $4.50 | — | 400,000 | 128,000 | Cost-optimized coding, computer use, sub-agents |
| **`gpt-5.4-nano`** | $0.20 | $1.25 | — | 400,000 | 128,000 | Speed/cost-critical: classification, extraction, ranking |
| **`gpt-5.3-codex`** | $1.75 | $14.00 | — | 400,000 | 128,000 | Agentic coding in Codex-style environments |

> **`reasoning_effort` default change (important).** GPT-5 defaulted to `medium`. GPT-5.1/5.2 flipped the default to `none`. **GPT-5.5 reverts the default back to `medium`** and adds a new top tier `xhigh`. Supported values for `gpt-5.5`: `none`, `low`, `medium`, `high`, `xhigh`.
>
> Because the default moved twice across the 5.x line (`medium` -> `none` -> `medium`), **always pin `reasoning_effort` explicitly** in production code so cost and latency are predictable. Don't assume "5.2's `none` is inherited by 5.5" — it is not.
>
> **Long-context surcharge:** prompts over 272K input tokens are billed at 2x input / 1.5x output for the full session.

> **`gpt-5.5-instant` is now available in the API.** Use `model="gpt-5.5-instant"` for fast, latency-sensitive cells. This notebook uses `gpt-5.5` throughout for consistency.

OpenAI's system card / marketing names differ from the API identifiers used in code. Current mapping (June 2026):

| System card / display name | API identifier | Notes |
|---|---|---|
| GPT-5.5 | `gpt-5.5` | Current flagship |
| GPT-5.5 pro | `gpt-5.5-pro` | Highest precision |
| GPT-5.4 | `gpt-5.4` | Cost-efficient coding |
| GPT-5.4 mini | `gpt-5.4-mini` | Strongest mini / subagents |
| GPT-5.4 nano | `gpt-5.4-nano` | High-volume, lowest cost |
| o3 | `o3` | Dedicated reasoning model |
| o3 pro | `o3-pro` | Extended reasoning |
| o4-mini | `o4-mini` | Fast reasoning |
| GPT-4.1 | `gpt-4.1` | Legacy workhorse |
| GPT-4.1 mini | `gpt-4.1-mini` | |
| GPT-4.1 nano | `gpt-4.1-nano` | |
| GPT Image 2 | `gpt-image-2` | Image generation |
| GPT-4.5 | `gpt-4.5-preview` | **Deprecated** |

> **Note:** GPT-5.x models expose `reasoning_effort` (`low` / `medium` / `high`) directly — they are not separate model SKUs like the old o-series.
>
> Source: [developers.openai.com/api/docs/models](https://developers.openai.com/api/docs/models/all)

## Quickstart: Your First GPT-5.5 Response

A minimal single-turn call using the Responses API. Use `reasoning={"effort": "low"}` for fast, low-cost outputs on simple tasks:

In [50]:
# Simple quickstart: low effort for fast, cheap responses on simple tasks
result = client.responses.create(
    model="gpt-5.5",
    input="Write a haiku about code.",
    reasoning={"effort": "low"},
)

print("Output:", result.output_text)
print("\nReasoning tokens used:", len(result.reasoning_text.split()) if hasattr(result, 'reasoning_text') else 'N/A')

## Reasoning Effort Control

GPT-5.5 supports three reasoning effort levels:

- **`low`**: Fastest time-to-first-token, lowest cost — best for simple tasks and coding
- **`medium`**: Default, balanced reasoning — good for most tasks
- **`high`**: Most thorough reasoning — best for complex problems, math, deep analysis

> **Note:** `minimal` was available in earlier GPT-5.x releases but is not supported in `gpt-5.5`. Use `low` as the fastest option.

In [ ]:
# Low reasoning for fastest response (gpt-5.5 does not support "minimal"; use "low" or "none")
response_minimal = client.responses.create(
    model="gpt-5.5",
    input="Write a Python function to check if a number is prime.",
    reasoning={"effort": "low"}
)

print("Low Reasoning Output:")
print(response_minimal.output_text)

In [5]:
# High reasoning for complex problems
response_high = client.responses.create(
    model="gpt-5.5",
    input="How much gold would it take to coat the Statue of Liberty in a 1mm layer? Show your calculations.",
    reasoning={"effort": "high"}
)

print("High Reasoning Output:")
print(response_high.output_text)

High Reasoning Output:
Short answer: about 25–30 metric tons of gold. A reasonable estimate using the statue’s known copper skin gives about 25.5 metric tons.

Assumptions and data
- Coating only the statue (not the pedestal), exterior surface only
- Copper skin thickness of the statue: 3/32 in = 0.09375 in = 0.00238125 m
- Mass of the statue’s copper skin: 62,000 lb = 28,124 kg
- Density of copper ρCu = 8,960 kg/m^3
- Density of gold ρAu = 19,320 kg/m^3
- Desired gold thickness tAu = 1 mm = 0.001 m

Steps
1) Infer the statue’s exterior surface area A from the copper skin:
   A = mCu / (ρCu × tCu)
     = 28,124 kg / (8,960 kg/m^3 × 0.00238125 m)
     ≈ 28,124 / 21.336
     ≈ 1,318 m^2

2) Volume of gold needed for a 1 mm coat:
   VAu = A × tAu = 1,318 m^2 × 0.001 m = 1.318 m^3

3) Mass of gold:
   mAu = ρAu × VAu = 19,320 kg/m^3 × 1.318 m^3 ≈ 25,458 kg

Result
- Gold required ≈ 25,500 kg ≈ 25.5 metric tons (≈ 28.1 short tons)

Note
- If you prefer to use a different surface-area estima

## Verbosity Control

Control output length with `verbosity` parameter:
- **`low`**: Concise answers, minimal code comments
- **`medium`**: Balanced explanations (default)
- **`high`**: Thorough explanations, detailed code documentation

In [52]:
# Low verbosity for concise responses
response_concise = client.responses.create(
    model="gpt-5.5",
    input="Generate a SQL query to find the top 5 customers by total purchase amount.",
    text={"verbosity": "low"}
)

print("Concise Output:")
print(response_concise.output_text)

Concise Output:
SELECT c.id AS customer_id,
       c.name AS customer_name,
       SUM(o.total_amount) AS total_purchase_amount
FROM customers c
JOIN orders o ON o.customer_id = c.id
GROUP BY c.id, c.name
ORDER BY total_purchase_amount DESC
LIMIT 5;  -- For SQL Server use: SELECT TOP 5 ... (and remove LIMIT)


In [ ]:
# High verbosity for detailed explanations
response_detailed = client.responses.create(
    model="gpt-5.5",
    input="Explain how async/await works in JavaScript.",
    text={"verbosity": "high"}
)

print("Detailed Output:")
print(response_detailed.output_text + "...")  # Truncated for display

In [55]:
print(len(response_concise.output_text))

print(len(response_detailed.output_text))

293
5334


In [58]:
output_cost_verbosity_low = 293*10/1000000
output_cost_verbosity_high = 5334*10/1000000

print("Ratio of cost for verbosity high vs low is")
print(output_cost_verbosity_high/output_cost_verbosity_low)

Ratio of cost for verbosity high vs low is
18.204778156996586


- High verbosity: Use when you need the model to provide thorough explanations of documents or perform extensive code refactoring.

- Low verbosity: Best for situations where you want concise answers or simple code generation, such as SQL queries.

## Custom Tools: Freeform Text Inputs

GPT-5 introduces **custom tools** that accept raw text instead of structured JSON. Perfect for:
- Executing code snippets
- SQL queries
- Shell commands
- Configuration files

In [59]:
# Define a custom tool for code execution
response_with_tool = client.responses.create(
    model="gpt-5.5",
    input="Use the code_exec tool to calculate the factorial of 10.",
    tools=[
        {
            "type": "custom",
            "name": "code_exec",
            "description": "Executes arbitrary Python code that prints something and returns the result"
        }
    ]
)

print("Tool Call Generated:")
for reasoning_output in response_with_tool.output:
    if reasoning_output.type == "custom_tool_call":
        print(f"Tool: {reasoning_output.name}")
        print(f"Input: {reasoning_output.input}")

Tool Call Generated:
Tool: code_exec
Input: import math
print(math.factorial(10))


In [60]:
def code_exec(code: str) -> str:
    """
    Executes arbitrary Python code and returns the result as a string.
    WARNING: This function is dangerous and should only be used in secure, sandboxed environments.
    """
    import sys
    import io
    import traceback

    # Redirect stdout to capture print statements
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    result = ""
    try:
        # Try to compile as an expression first
        try:
            compiled = compile(code, "<string>", "eval")
            output = eval(compiled, {}, {})
            if output is not None:
                print(output)
        except SyntaxError:
            # If not an expression, treat as statements
            exec(code, {}, {})
        result = sys.stdout.getvalue()
    except Exception:
        result = "Error:\n" + traceback.format_exc()
    finally:
        sys.stdout = old_stdout
    return result.strip()

code_exec(reasoning_output.input)

'3628800'

## Context-Free Grammar (CFG) Constraints

Constrain custom tool outputs to specific syntax using Lark grammars (or regex grammrs). 

We'll see a detailed example in notebook 2.0.

## Allowed Tools: Selective Tool Access

Define a full toolkit but restrict which tools can be used in specific contexts:

In [34]:
# Define multiple tools but restrict usage
all_tools = [
    {"type": "function", "name": "get_weather", "description": "Get current weather"},
    {"type": "function", "name": "search_docs", "description": "Search documentation"},
    {"type": "function", "name": "run_tests", "description": "Execute test suite"},
    {"type": "function", "name": "deploy_code", "description": "Deploy to production"}
]

# Only allow safe operations
response_restricted = client.responses.create(
    model="gpt-5.5",
    input="What's the weather like and can you search for React hooks documentation?",
    tools=all_tools,
    tool_choice={
        "type": "allowed_tools",
        "mode": "auto",  # Model decides which to use
        "tools": [
            {"type": "function", "name": "get_weather"},
            {"type": "function", "name": "search_docs"}
        ]
    }
)

response_restricted.output

[ResponseReasoningItem(id='rs_68d691511d5081978f965991433591860257326b2b51fa3d', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{}', call_id='call_jmfCsbT4SMmwhJpplgvZojSl', name='get_weather', type='function_call', id='fc_68d69155d3148197a52b0a1e93bd5dfb0257326b2b51fa3d', status='completed'),
 ResponseFunctionToolCall(arguments='{}', call_id='call_PArxuLpuaC7kF6Bq01RPvkMr', name='search_docs', type='function_call', id='fc_68d69155e72081979fbeedce747ba4230257326b2b51fa3d', status='completed')]

In [44]:
response_restricted.tool_choice.tools

[{'type': 'function', 'name': 'get_weather'},
 {'type': 'function', 'name': 'search_docs'}]

## Tool Preambles for Transparency

Enable preambles to see GPT-5's reasoning before tool calls:

In [45]:
# Enable preambles for tool transparency
response_preamble = client.responses.create(
    model="gpt-5.5",
    input="Before you call a tool, explain why you are calling it. Now search for information about Python decorators.",
    tools=[
        {"type": "function", "name": "search_docs", "description": "Search documentation"}
    ]
)

print("Response with preamble:")
print(response_preamble.output_text)

Response with preamble:
I will use the search tool to fetch authoritative, up-to-date references and tutorials on Python decorators so I can provide accurate and concise information.


## Migration Guide

### From Other Models to GPT-5.5

| From Model | Migrate To | Recommended Settings |
|------------|------------|---------------------|
| **o3** | `gpt-5.5` | `reasoning.effort: "medium"` or `"high"` |
| **gpt-4.1** | `gpt-5.5` | `reasoning.effort: "low"` |
| **o4-mini** | `gpt-5.4-mini` | Default settings with prompt tuning |
| **gpt-4.1-nano** | `gpt-5.4-nano` | Default settings with prompt tuning |

### ⚠️ Important: Unsupported Parameters

GPT-5.5 does **NOT** support:
- `temperature`
- `top_p`
- `logprobs`

Use GPT-5.5-specific controls instead:
- `reasoning: {effort: ...}`
- `text: {verbosity: ...}`
- `max_output_tokens`

## Responses API vs Chat Completions

### Key Advantage: Chain of Thought (CoT) Persistence

The Responses API passes reasoning between turns, resulting in:
- Improved intelligence
- Fewer reasoning tokens generated
- Higher cache hit rates
- Lower latency

In [46]:
# Multi-turn conversation with the Conversations API (current pattern)
# client.conversations.create() makes a durable conversation object that stores
# messages, tool calls and tool outputs. Pass its id on every turn and the model
# pulls in the shared context automatically -- no manual history rebuilding.

conversation = client.conversations.create()

first_response = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="Let's solve a complex problem. What's the optimal way to implement an LRU cache in Python?",
    reasoning={"effort": "medium"},
)
print("First response:", first_response.output_text[:200] + "...")

# Continue the SAME conversation -- context is shared via the conversation id.
follow_up = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="Now add thread-safety to that implementation.",
    reasoning={"effort": "medium"},
)
print("\nFollow-up (shared conversation context):", follow_up.output_text[:200] + "...")

First response: “Optimal” depends on what you need:

- Fastest and simplest for function memoization: use functools.lru_cache (implemented in C).
- General-purpose key/value LRU cache with O(1) get/put: use OrderedDi...

Follow-up (with CoT context): Here’s a thread-safe LRU cache (O(1) get/put) built on OrderedDict with an internal RLock. It snapshots iteration results to avoid races during traversal.

from collections import OrderedDict
import t...


> **Legacy note — `previous_response_id`.** Before the Conversations API, you chained turns by passing `previous_response_id=first_response.id` on the next `responses.create()` call. That still works for `gpt-5.5`, but `client.conversations.create()` is now the recommended pattern:
>
> ```python
> # Legacy chaining (still supported, but prefer conversations):
> follow_up = client.responses.create(
>     model="gpt-5.5",
>     input="Now add thread-safety to that implementation.",
>     previous_response_id=first_response.id,  # pulls CoT from the previous turn
> )
> ```
>
> Billing caveat (both patterns): every prior input token in the chain is re-billed as input on each turn. Responses attached to a conversation persist indefinitely (no 30-day TTL), unlike `store: true` responses which expire after 30 days.

## Best Practices

### 1. Choose the Right Model
- **`gpt-5.5`**: Frontier coding, complex reasoning, multi-step professional tasks
- **`gpt-5.4-mini`**: Subagents, moderate tasks, cost-sensitive workloads
- **`gpt-5.4-nano`**: High-volume simple tasks, lowest cost

### 2. Optimize for Your Use Case
- **Speed Priority**: Use `low` reasoning + `low` verbosity
- **Quality Priority**: Use `high` reasoning + `high` verbosity
- **Balanced**: Use defaults (`medium` for both)

### 3. Leverage New Features
- **Custom Tools**: For code execution, SQL, configs
- **Allowed Tools**: For safety and predictability
- **Preambles**: For debugging and transparency

### 4. Use Conversations API for Multi-Turn
- Create a durable conversation with `client.conversations.create()`
- Pass `conversation=conversation.id` on each turn — no manual history rebuilding
- `previous_response_id` still works but conversations are now the recommended pattern

## Practical Example: Building a Code Assistant

In [ ]:
def code_assistant(task, code_context=None, optimize_for="balanced"):
    """
    GPT-5 powered code assistant with configurable optimization.
    
    Args:
        task: What you want the assistant to do
        code_context: Existing code to work with
        optimize_for: "speed", "quality", or "balanced"
    """
    
    # Configure based on optimization preference
    settings = {
        "speed": {"reasoning": {"effort": "low"}, "text": {"verbosity": "low"}},
        "quality": {"reasoning": {"effort": "high"}, "text": {"verbosity": "high"}},
        "balanced": {"reasoning": {"effort": "medium"}, "text": {"verbosity": "medium"}}
    }
    
    config = settings.get(optimize_for, settings["balanced"])
    
    # Build the prompt
    prompt = task
    if code_context:
        prompt = f"Task: {task}\n\nExisting code:\n```python\n{code_context}\n```"
    
    # Call GPT-5
    response = client.responses.create(
        model="gpt-5.5",
        input=prompt,
        **config
    )
    
    return response.output_text

# Example usage
result = code_assistant(
    task="Create a fun html app that could be cool to visualize inside a jupyter notebook, get creative and minimalistic.",
    code_context="""""",
    optimize_for="speed"
)

print(result)

## Conclusion

GPT-5.5 represents a significant leap in AI reasoning capabilities. Key takeaways:

1. **Use the Responses API** for multi-turn conversations via `client.conversations.create()`
2. **Configure reasoning and verbosity** based on your latency/quality requirements
3. **Leverage new features** like custom tools and allowed tools for better control
4. **Choose the right model variant** (`gpt-5.5`, `gpt-5.4-mini`, `gpt-5.4-nano`) for your use case
5. **Migrate gradually** using the recommended settings for your current model

For more information:
- [All Models](https://developers.openai.com/api/docs/models/all)
- [GPT-5 Prompting Guide](https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide)
- [API Changelog](https://developers.openai.com/api/docs/changelog)